In [0]:
#Data Quality Bronze  (apenas informativo, visto que toda carga é aceita)

import requests
import json
from pyspark.sql.functions import (current_timestamp, lit, max)
import pandas as pd


bronze_df = spark.read.table("curso_eng_dados.modulo_um.raw_randomuser_data")

ultima_datahora = bronze_df.agg({"dt_ingestao": "max"}).collect()[0][0]

bronze_df = bronze_df.filter(bronze_df.dt_ingestao == ultima_datahora)


def valida_bronze(bronze_df):

    total_linhas = bronze_df.count()

    nulos = bronze_df.filter(
        bronze_df.payload.isNull()
    ).count()

    dt_ingestao = bronze_df.select(
        "dt_ingestao"
    ).first()[0]
    status = "PASS"

    if total_linhas == 0:
        status = "FAIL"

    audit_data = [{
        "pipeline": "api_random",
        "source": "RandomUser",
        "dt_ingestao": dt_ingestao,
        "rule_name": "payload_not_null",
        "total_linhas": total_linhas,
        "status": status,
        "missed": nulos
    }]

    df_audit = spark.createDataFrame(audit_data)

    df_audit.write \
        .mode("append") \
        .saveAsTable(
            "curso_eng_dados.modulo_um.raw_randomuser_dq"
        )

    return bronze_df

valida_bronze(bronze_df)